In [10]:
import pandas as pd
import numpy as np

# Задание №1

**Генератор признаков**

In [11]:
def generate_features(n_samples, n_features, prefix):
    df = pd.DataFrame()
    
    nominal = [1000, 500, 333, 777]
    ordinal = [1, 2, 3, 4]
    
    for i in range(1, n_features + 1):
        if i % 4 == 1:
            df[f'{prefix}_bin_{i}'] = np.random.choice([1, 0], n_samples)
        elif i % 4 == 2:
            df[f'{prefix}_nominal_{i}'] = np.random.choice(nominal, n_samples)
        elif i % 4 == 3:
            df[f'{prefix}_ordinal_{i}'] = np.random.choice(ordinal, n_samples)
        else:
            df[f'{prefix}_quantitative_{i}'] = np.random.uniform(10, 1000, n_samples)
            
    return df


**Функция коллизии**

In [12]:
def collision(row1, row2):
    diff_sum = 0
    v1 = row1.values
    v2 = row2.values
    cols = row1.index 
    
    for i in range(len(cols)):
        col_name = cols[i]
        val1 = v1[i]
        val2 = v2[i]
        
        if 'bin' in col_name or 'nominal' in col_name:
            diff_sum += 1 if val1 != val2 else 0
        elif 'ordinal' in col_name:
            if val1 == 777 or val2 == 777:
                diff_sum += 2
            elif val1 == 333 or val2 == 333:
                diff_sum += 1
        else:
            diff_sum += abs(val1 - val2) / 1000
    
    return 1 if diff_sum < len(cols) * 0.4 else 0

**Генерация датасетов**

In [13]:
def create_dataset(n_samples, n_features_obj1, n_features_obj2):
    obj1_df = generate_features(n_samples, n_features_obj1, 'Obj1')
    obj2_df = generate_features(n_samples, n_features_obj2, 'Obj2')
    
    dataset = pd.concat([obj1_df, obj2_df], axis=1)
    
    results = []
    for i in range(len(dataset)):
        res = collision(obj1_df.iloc[i], obj2_df.iloc[i])
        results.append(res)

    dataset['collision'] = results
    
    return dataset


In [14]:
sample_sizes = [50, 250, 750, 1500]
feature_counts = [5, 9, 15]
datasets = {}
dataset_id = 1

for n_samples in sample_sizes:
    for n_features in feature_counts:
        df = create_dataset(n_samples, n_features, n_features)
        name = f"ID{dataset_id}_S{n_samples}_F{n_features}"
        datasets[name] = df
        dataset_id += 1
        df.to_csv(name)

for ds in datasets:
    print(ds)

datasets["ID3_S50_F15"].head()

ID1_S50_F5
ID2_S50_F9
ID3_S50_F15
ID4_S250_F5
ID5_S250_F9
ID6_S250_F15
ID7_S750_F5
ID8_S750_F9
ID9_S750_F15
ID10_S1500_F5
ID11_S1500_F9
ID12_S1500_F15


,Obj1_bin_1,Obj1_nominal_2,Obj1_ordinal_3,Obj1_quantitative_4,Obj1_bin_5,Obj1_nominal_6,Obj1_ordinal_7,Obj1_quantitative_8,Obj1_bin_9,Obj1_nominal_10,...,Obj2_ordinal_7,Obj2_quantitative_8,Obj2_bin_9,Obj2_nominal_10,Obj2_ordinal_11,Obj2_quantitative_12,Obj2_bin_13,Obj2_nominal_14,Obj2_ordinal_15,collision
0,1,500,3,918.354580,1,1000,2,479.924966,1,333,...,3,572.847809,1,333,2,231.118244,1,777,4,1
1,0,333,2,712.215937,0,500,2,558.201961,0,333,...,1,398.458980,1,1000,2,305.994629,0,777,4,0
2,1,500,3,539.292812,0,500,4,965.923765,1,1000,...,3,17.789278,0,333,2,175.729991,1,500,1,1
3,0,1000,1,217.554257,0,1000,1,651.093597,0,333,...,3,934.579460,1,500,1,203.978600,1,333,3,1
4,1,1000,2,773.552774,0,500,2,806.445766,0,1000,...,1,700.394850,0,333,3,701.447895,0,777,3,1


# Задание №2

In [15]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def evaluate_model_on_datasets(model, datasets):
    model_name = model.__class__.__name__
    results = {'Method': model_name}
    
    for i, df in enumerate(datasets):
        y = df["collision"]
        X = df.drop(["collision"], axis=1)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=228)

        feature_names = X_train.columns
        scaler = StandardScaler().fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
        X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names)
        
        model.fit(X_train_scaled, y_train)
        
        y_pred = model.predict(X_test_scaled)
        score = f1_score(y_test, y_pred, average='weighted')
        
        results[f'Dataset_{i+1}'] = score
        
    return pd.Series(results)

In [16]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [17]:
models = [
    KNeighborsClassifier(),
    GaussianNB(),
    RandomForestClassifier(),
    LogisticRegression()
]

rows = []
for model in models:
    row = evaluate_model_on_datasets(model, list(datasets.values()))
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df['average'] = results_df.mean(axis=1, numeric_only=True)
results_df

,Method,Dataset_1,Dataset_2,Dataset_3,Dataset_4,Dataset_5,Dataset_6,Dataset_7,Dataset_8,Dataset_9,Dataset_10,Dataset_11,Dataset_12,average
0,KNeighborsClassifier,0.525000,0.800000,0.466667,0.786731,0.551429,0.560000,0.841807,0.691690,0.550966,0.844612,0.653179,0.660060,0.661012
1,GaussianNB,0.321212,0.505051,0.516484,0.599500,0.504236,0.580504,0.474568,0.552078,0.585629,0.459085,0.542776,0.522104,0.513602
2,RandomForestClassifier,0.525000,0.780952,0.600000,0.673779,0.636522,0.554220,0.846054,0.572877,0.605342,0.872471,0.688891,0.622834,0.664912
3,LogisticRegression,0.466667,0.600000,0.600000,0.566420,0.462923,0.539071,0.474568,0.486598,0.553830,0.460664,0.542154,0.492202,0.520425


In [18]:
from sklearn.model_selection import GridSearchCV

param_grids = [
    {
        'n_neighbors': [1, 2, 3, 5, 7, 9, 11, 13, 15],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski'],
        'p': [1, 1.5, 2]
    },
    {
        'var_smoothing': np.logspace(-10, -1, num=10)
    },
    {
        'n_estimators': [100, 200, 300],
        'min_samples_split': [2, 5, 10],
        'random_state': [None, 777],
        'bootstrap': [True, False]
    },
    {
        'l1_ratio': [0, 0.3, 0.5, 0.7, 1],
        'C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'solver': ['lbfgs', 'liblinear', 'newton-cg', 'saga'],
        'random_state': [None, 777]
    }
]

tuned_rows = []

for model, grid, df in zip(models, param_grids, list(datasets.values())):
    grid_search = GridSearchCV(model, grid, cv=5, scoring='f1_weighted', n_jobs=-1)
    row = evaluate_model_on_datasets(grid_search, list(datasets.values()))
    tuned_rows.append(row)

tuned_results_df = pd.DataFrame(tuned_rows)
tuned_results_df['average'] = tuned_results_df.mean(axis=1, numeric_only=True)
tuned_results_df

c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
550 fits failed out of a total of 1000.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
150 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\admin\Ap

,Method,Dataset_1,Dataset_2,Dataset_3,Dataset_4,Dataset_5,Dataset_6,Dataset_7,Dataset_8,Dataset_9,Dataset_10,Dataset_11,Dataset_12,average
0,GridSearchCV,0.600,0.703030,0.525000,0.804365,0.598718,0.640577,0.842760,0.726484,0.593713,0.878662,0.696940,0.693142,0.691949
1,GridSearchCV,0.400,0.505051,0.516484,0.599500,0.504236,0.580504,0.474568,0.552078,0.585629,0.460664,0.544835,0.522104,0.520471
2,GridSearchCV,0.680,0.450000,0.400000,0.703501,0.657953,0.579152,0.866537,0.691023,0.580392,0.878980,0.729471,0.656380,0.656116
3,GridSearchCV,0.525,0.483516,0.600000,0.577561,0.462923,0.560000,0.474568,0.453333,0.553830,0.460664,0.542154,0.498525,0.516006


In [19]:
results_df

,Method,Dataset_1,Dataset_2,Dataset_3,Dataset_4,Dataset_5,Dataset_6,Dataset_7,Dataset_8,Dataset_9,Dataset_10,Dataset_11,Dataset_12,average
0,KNeighborsClassifier,0.525000,0.800000,0.466667,0.786731,0.551429,0.560000,0.841807,0.691690,0.550966,0.844612,0.653179,0.660060,0.661012
1,GaussianNB,0.321212,0.505051,0.516484,0.599500,0.504236,0.580504,0.474568,0.552078,0.585629,0.459085,0.542776,0.522104,0.513602
2,RandomForestClassifier,0.525000,0.780952,0.600000,0.673779,0.636522,0.554220,0.846054,0.572877,0.605342,0.872471,0.688891,0.622834,0.664912
3,LogisticRegression,0.466667,0.600000,0.600000,0.566420,0.462923,0.539071,0.474568,0.486598,0.553830,0.460664,0.542154,0.492202,0.520425


In [20]:
import time

def get_train_time(model, X, y):
    start_time = time.time()
    model.fit(X, y)
    end_time = time.time()
    
    return end_time - start_time

In [21]:
datasets_to_run = {
    "Small": datasets["ID1_S50_F5"],
    "Big": datasets["ID12_S1500_F15"]
}

fit_results = []

for model in models:
    model_name = model.__class__.__name__
    row = {"Model": model_name}
    
    for ds_name, df in datasets_to_run.items():
        y = df["collision"]
        X = df.drop(["collision"], axis=1)
        
        X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=228)

        row[ds_name] = get_train_time(model, X_train, y_train)
    
    fit_results.append(row)

df_results = pd.DataFrame(fit_results)
df_results

c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://sciki

,Model,Small,Big
0,KNeighborsClassifier,0.001366,0.001418
1,GaussianNB,0.001426,0.001939
2,RandomForestClassifier,0.102146,0.277143
3,LogisticRegression,0.018121,0.020958


In [22]:
import joblib

datasets_to_run = {
    "Small": datasets["ID1_S50_F5"],
    "Big": datasets["ID12_S1500_F15"]
}

param_grids = [
    {
        'n_neighbors': [1, 2, 3, 5, 7, 9, 11, 13, 15],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski'],
        'p': [1, 1.5, 2]
    },
    {
        'var_smoothing': np.logspace(-10, -1, num=10)
    },
    {
        'n_estimators': [100, 200, 300],
        'min_samples_split': [2, 5, 10],
        'random_state': [None, 777],
        'bootstrap': [True, False]
    },
    {
        'l1_ratio': [0, 0.3, 0.5, 0.7, 1],
        'C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'solver': ['lbfgs', 'liblinear', 'newton-cg', 'saga'],
        'random_state': [None, 777]
    }
]

fit_results = []

for i, model in enumerate(models):
    model_name = model.__class__.__name__
    row = {"Model": model_name}
    
    for ds_name, df in datasets_to_run.items():
        y = df["collision"]
        X = df.drop(["collision"], axis=1)
        
        X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=228)
        grid_search = GridSearchCV(model, param_grids[i], cv=5, scoring='f1_weighted', n_jobs=-1)
        
        row[ds_name] = get_train_time(grid_search, X_train, y_train)
    
    joblib.dump(model, model_name)
        
    
    fit_results.append(row)

df_results = pd.DataFrame(fit_results)
df_results

c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
550 fits failed out of a total of 1000.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
150 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\admin\Ap

,Model,Small,Big
0,KNeighborsClassifier,0.610465,2.531045
1,GaussianNB,0.080164,0.103165
2,RandomForestClassifier,4.116801,10.344154
3,LogisticRegression,0.727083,1.720691
